# GNN-RNN Hybrid Model Lifecycle

This notebook walks through the **full lifecycle** for the **Hybrid GNN-RNN** model: predict **target trade P&L** from elementary P&L time series, an adjacency matrix over trades, and encoded trade features. The same design as the generic ML lifecycle applies: **data** → **model** → **train** → **evaluate** → **inference**, with model-specific data and model only in `data/gnn_rnn_hybrid` and `models/gnn_rnn_hybrid`.

## What you'll see

| Step | Component | Purpose |
|------|------------|--------|
| 1 | **Data** | `data/gnn_rnn_hybrid.build_gnn_data()` → `train_ds`, `val_ds`, `proj_ds` (tf.data.Dataset) |
| 2 | **Model** | `models/gnn_rnn_hybrid`: BatchedHybridGnnRnn + `default_hybrid_model_config` |
| 3 | **Train** | Keras `model.fit(train_ds, validation_data=val_ds)` |
| 4 | **Evaluate** | Metrics on projection set; standardised evaluation pattern |
| 5 | **Inference** | Predict on new batches from the same data interface |

Batching is defined by the data builder; the model consumes the batched Dataset.

---
## 0. Configuration

Data size, splits, training hyperparameters, and model config. Use synthetic data for a fast run; set `USE_SYNTHETIC = False` for FX portfolio data (slower).

In [ ]:
USE_SYNTHETIC = True
N_TRADES = 50
N_ELEMENTARY = 30
N_TARGETS = 10
N_SAMPLES = 400
N_TIMESTEPS = 20
K_NEIGHBOURS = 5
TRAIN_RATIO, VAL_RATIO, PROJ_RATIO = 0.6, 0.2, 0.2
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-3
SEED = 42

---
## 1. Data

Use the **per-model data builder** for GNN-RNN: `build_gnn_data()` returns `train_ds`, `val_ds`, `proj_ds` as `tf.data.Dataset`. Synthetic or FX portfolio data is selected via `use_synthetic`.

In [ ]:
from src.m_learning.data.gnn_rnn_hybrid import build_gnn_data

data = build_gnn_data(
    use_synthetic=USE_SYNTHETIC,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    projection_ratio=PROJ_RATIO,
    batch_size=BATCH_SIZE,
    seed=SEED,
    n_trades=N_TRADES,
    n_elementary=N_ELEMENTARY,
    n_targets=N_TARGETS,
    n_samples=N_SAMPLES,
    n_timesteps=N_TIMESTEPS,
    k_neighbours=K_NEIGHBOURS,
    noise_std=0.5,
)

print("Metadata:", data.metadata)
print("Train batches:", len(list(data.train_ds)))

---
## 2. Model

Build the **GNN-RNN model** from `models/gnn_rnn_hybrid`: config from `default_hybrid_model_config`, then `BatchedHybridGnnRnn`. One model class; batching comes from the data.

In [ ]:
import tensorflow as tf
from src.m_learning.models.gnn_rnn_hybrid import (
    BatchedHybridGnnRnn,
    default_hybrid_model_config,
)

model_config = default_hybrid_model_config(
    gnn_units=32,
    rnn_units=32,
    fusion_units=32,
    attention_units=32,
    projection_units=32,
    n_targets=N_TARGETS,
)
model = BatchedHybridGnnRnn(model_config, name="hybrid_pnl")
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss="mse",
    metrics=["mae"],
)
model.summary()

---
## 3. Train

Train with Keras using the datasets from step 1.

In [ ]:
history = model.fit(
    data.train_ds,
    validation_data=data.val_ds,
    epochs=EPOCHS,
    verbose=1,
)
print("Final val loss:", history.history["val_loss"][-1])

---
## 4. Evaluate

Evaluate on the projection set: collect predictions and compute MSE, MAE, R². Same standardised pattern as the generic lifecycle.

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_true_list = []
y_pred_list = []
for inputs, targets in data.proj_ds:
    preds = model(inputs, training=False)
    y_true_list.append(targets.numpy())
    y_pred_list.append(preds.numpy())
y_true = np.vstack(y_true_list)
y_pred = np.vstack(y_pred_list)

mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print("Projection set — MSE: {:.6f}, MAE: {:.6f}, R²: {:.4f}".format(mse, mae, r2))

---
## 5. Inference

Run inference on a batch from the same data interface. In production you would load a saved model and feed new (inputs, targets) from your data pipeline.

In [ ]:
for inputs, _ in data.proj_ds.take(1):
    predictions = model(inputs, training=False)
    print("Batch predictions shape:", predictions.shape)
    break

---
## Summary

| Step | Module | Output |
|------|--------|--------|
| Data | `data.gnn_rnn_hybrid.build_gnn_data` | `GnnDataResult(train_ds, val_ds, proj_ds)` |
| Model | `models.gnn_rnn_hybrid.BatchedHybridGnnRnn` + `default_hybrid_model_config` | Keras model |
| Train | `model.fit(data.train_ds, validation_data=data.val_ds)` | History |
| Evaluate | Projection set metrics (MSE, MAE, R²) | Standardised metrics |
| Inference | `model(inputs, training=False)` | Predictions tensor |

The same pipeline pattern as the pricing lifecycle; only the data builder and model are GNN-RNN–specific.